In [1]:
# !pip install -q -U \
#   torch \
#   torchvision \
#   torchaudio \
#   transformers \
#   datasets \
#   accelerate \
#   peft \
#   trl \
#   bitsandbytes \
#   sentencepiece \
#   safetensors \
#   evaluate \
#   scikit-learn

In [2]:
# pip install cuda-toolkit==12.6.0

In [3]:
# pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

In [4]:
from pathlib import Path
import sys
import os
import json
import time
import torch
import pandas as pd


In [5]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

from transformers import AutoTokenizer
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [6]:
if os.path.exists('/content'):
    PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path(".")

DATA_DIR = PROJECT_ROOT / "data"
SFT_DIR = DATA_DIR / "sft_ready"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lora_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL = SFT_DIR / "feina_30_train_sft.jsonl"
VAL_JSONL   = SFT_DIR / "feina_30_val_sft.jsonl"
TEST_JSONL  = SFT_DIR / "feina_30_test_sft.jsonl"

print("TRAIN_JSONL:", TRAIN_JSONL)
print("VAL_JSONL  :", VAL_JSONL)
print("TEST_JSONL :", TEST_JSONL)

TRAIN_JSONL: /content/data/sft_ready/feina_30_train_sft.jsonl
VAL_JSONL  : /content/data/sft_ready/feina_30_val_sft.jsonl
TEST_JSONL : /content/data/sft_ready/feina_30_test_sft.jsonl


In [7]:
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb
import accelerate

print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("peft        :", peft.__version__)
print("trl         :", trl.__version__)
print("accelerate  :", accelerate.__version__)
print("bnb         :", bnb.__version__)
print("torch       :", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

transformers: 5.5.3
datasets    : 4.8.4
peft        : 0.18.1
trl         : 1.1.0
accelerate  : 1.13.0
bnb         : 0.49.2
torch       : 2.5.1+cu121
cuda disponible: True
gpu: Tesla T4


In [8]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
RUN_NAME = "lora_llama3_feina30_v1"

print("MODEL_ID:", MODEL_ID)
print("RUN_NAME:", RUN_NAME)

MODEL_ID: meta-llama/Meta-Llama-3-8B-Instruct
RUN_NAME: lora_llama3_feina30_v1


In [9]:
from datasets import load_dataset

data_files = {
    "train": str(TRAIN_JSONL),
    "validation": str(VAL_JSONL),
    "test": str(TEST_JSONL),
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)
print(dataset["train"][0].keys())
print(dataset["train"][0]["text"][:500])

DatasetDict({
    train: Dataset({
        features: ['row_id', 'instruction', 'output', 'text'],
        num_rows: 1110
    })
    validation: Dataset({
        features: ['row_id', 'instruction', 'output', 'text'],
        num_rows: 238
    })
    test: Dataset({
        features: ['row_id', 'instruction', 'output', 'text'],
        num_rows: 238
    })
})
dict_keys(['row_id', 'instruction', 'output', 'text'])
Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.

Devuelve solo la versión final simplificada.

Texto:
Para la elaboración del documento se integró un equipo profesional interdisciplinario de trabajo con énfasis en administración de empresas, economía, planificación y educación, que tuvo a su cargo la investigación previa sobre el tema, la caracterización de la población meta o destinataria de la capacitación, 


In [10]:
from transformers import BitsAndBytesConfig

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("compute_dtype:", compute_dtype)
bnb_config

compute_dtype: torch.float16


BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

pad_token: <|eot_id|>
eos_token: <|eot_id|>


In [12]:
from transformers import AutoModelForCausalLM

t0 = time.perf_counter()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

t1 = time.perf_counter()
print(f"Tiempo cargando modelo: {t1 - t0:.2f} s")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Tiempo cargando modelo: 98.08 s


In [13]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
print("Modelo preparado para k-bit training")

Modelo preparado para k-bit training


In [14]:
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

print(peft_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'o_proj', 'k_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)


In [15]:
MAX_SEQ_LENGTH = 1024
MAX_SEQ_LENGTH

1024

In [16]:
from transformers import TrainingArguments

RUN_DIR = OUTPUT_DIR / RUN_NAME

training_args = TrainingArguments(
    output_dir=str(RUN_DIR),

    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.0,
    warmup_steps=50,
    lr_scheduler_type="cosine",

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    fp16=False,
    bf16=False,

    max_grad_norm=0.3,

    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

In [17]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
    formatting_func=lambda ex: ex["text"],
)

print("Trainer creado correctamente")

Trainer creado correctamente


In [18]:
t0 = time.perf_counter()

train_result = trainer.train()

t1 = time.perf_counter()
print(f"Tiempo total entrenamiento: {(t1 - t0)/60:.2f} min")
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss
1,0.933409,0.893489
2,0.862923,0.885466


Tiempo total entrenamiento: 45.63 min
TrainOutput(global_step=278, training_loss=1.039039179575529, metrics={'train_runtime': 2737.2132, 'train_samples_per_second': 0.811, 'train_steps_per_second': 0.102, 'total_flos': 1.278780091072512e+16, 'train_loss': 1.039039179575529})


In [19]:
FINAL_DIR = RUN_DIR / "final_adapter"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Adapter guardado en:", FINAL_DIR)

Adapter guardado en: /content/outputs/lora_runs/lora_llama3_feina30_v1/final_adapter


In [20]:
run_metadata = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "train_rows": len(dataset["train"]),
    "val_rows": len(dataset["validation"]),
    "test_rows": len(dataset["test"]),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "learning_rate": 2e-4,
    "epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,
    "compute_dtype": str(compute_dtype),
}

meta_path = RUN_DIR / "run_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)

print("Metadata guardada en:", meta_path)

Metadata guardada en: /content/outputs/lora_runs/lora_llama3_feina30_v1/run_metadata.json


In [21]:
sample_text = dataset["validation"][0]["instruction"]

inputs = tokenizer(sample_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.

Devuelve solo la versión final simplificada.

Texto:
Por otro lado, los activos netos son los activos que se tienen, descontadas o deducidas las deudas, por ejemplo, hipotecas.

Versión simplificada:Por otro lado, los activos netos son los activos que se tienen, descontadas o deducidas las deudas.
